# NEXTBUY: Cart Size Prediction Model

## Part 5: ML Model - Regression

This notebook builds a model to predict the average cart size per order.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')

## Load Data

In [ ]:
PROCESSED_PATH = Path("processed/")
parquet_file = PROCESSED_PATH / "full_data_engineered.parquet"
fallback_file = Path("full_data_engineered.parquet")

if parquet_file.exists():
    print("Loading from processed folder...")
    full_data = pd.read_parquet(parquet_file)
elif fallback_file.exists():
    print("Loading from current directory...")
    full_data = pd.read_parquet(fallback_file)
else:
    raise FileNotFoundError(
        "Engineered data not found. "
        "Run feature engineering notebook first to create full_data_engineered.parquet"
    )

print(f"Loaded: {full_data.shape}")
print(f"Columns: {list(full_data.columns)}")

## Feature Engineering

In [ ]:
# Create order-level features
order_features = full_data.groupby('order_id').agg({
    'user_id': 'first',
    'product_id': 'count',  # Target: cart_size
    'order_hour_of_day': 'first',
    'order_dow': 'first',
    'days_since_prior_order': 'first',
    'order_number': 'first',
    'reordered': 'mean',
    'order_frequency': 'first',
    'avg_basket_size': 'first'
}).reset_index()

# Additional features
order_features['is_weekend'] = order_features['order_dow'].isin([0, 6]).astype(int)

# Department diversity
dept_diversity = full_data.groupby('order_id')['department'].nunique().reset_index()
dept_diversity.columns = ['order_id', 'department_diversity']
order_features = order_features.merge(dept_diversity, on='order_id', how='left')

# Aisle diversity
aisle_diversity = full_data.groupby('order_id')['aisle'].nunique().reset_index()
aisle_diversity.columns = ['order_id', 'aisle_diversity']
order_features = order_features.merge(aisle_diversity, on='order_id', how='left')

# User total orders
user_order_count = full_data.groupby('user_id')['order_id'].nunique().reset_index()
user_order_count.columns = ['user_id', 'user_total_orders']
order_features = order_features.merge(user_order_count, on='user_id', how='left')

# Order ratio
order_features['order_ratio'] = order_features['order_number'] / order_features['user_total_orders']

order_features.rename(columns={'product_id': 'cart_size'}, inplace=True)

print(f"Order features shape: {order_features.shape}")
print(f"Features: {list(order_features.columns)}")

## Prepare Data for Modeling

In [ ]:
# Select features
feature_cols = [
    'order_hour_of_day', 'order_dow', 'days_since_prior_order',
    'order_number', 'reordered', 'order_frequency', 'avg_basket_size',
    'is_weekend', 'department_diversity', 'aisle_diversity',
    'user_total_orders', 'order_ratio'
]

model_data = order_features[feature_cols + ['cart_size']].dropna()

X = model_data[feature_cols]
y = model_data['cart_size']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Features: {feature_cols}")

## Train Models

In [ ]:
# Model 1: Linear Regression
print("Training Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_r2 = r2_score(y_test, y_pred_lr)

print(f"Linear Regression - MAE: {lr_mae:.2f}, RMSE: {lr_rmse:.2f}, R²: {lr_r2:.4f}")

In [ ]:
# Model 2: Random Forest
print("Training Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2 = r2_score(y_test, y_pred_rf)

print(f"Random Forest - MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}, R²: {rf_r2:.4f}")

In [ ]:
# Model 3: Gradient Boosting
print("Training Gradient Boosting...")
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

gb_mae = mean_absolute_error(y_test, y_pred_gb)
gb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb))
gb_r2 = r2_score(y_test, y_pred_gb)

print(f"Gradient Boosting - MAE: {gb_mae:.2f}, RMSE: {gb_rmse:.2f}, R²: {gb_r2:.4f}")

## Model Comparison

In [ ]:
models = ['Linear Regression', 'Random Forest', 'Gradient Boosting']
mae_scores = [lr_mae, rf_mae, gb_mae]
rmse_scores = [lr_rmse, rf_rmse, gb_rmse]
r2_scores = [lr_r2, rf_r2, gb_r2]

comparison = pd.DataFrame({
    'Model': models,
    'MAE': mae_scores,
    'RMSE': rmse_scores,
    'R²': r2_scores
})

print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)
print(comparison.to_string(index=False))

best_idx = r2_scores.index(max(r2_scores))
print(f"\nBest model: {models[best_idx]} (R² = {r2_scores[best_idx]:.4f})")

## Feature Importance

In [ ]:
importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['Feature'], importance['Importance'], color='teal')
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop features:")
for _, row in importance.iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.4f}")

## Prediction Example

In [ ]:
new_order = pd.DataFrame({
    'order_hour_of_day': [10],
    'order_dow': [2],
    'days_since_prior_order': [7],
    'order_number': [5],
    'reordered': [0.6],
    'order_frequency': [7.0],
    'avg_basket_size': [8.0],
    'is_weekend': [0],
    'department_diversity': [3],
    'aisle_diversity': [5],
    'user_total_orders': [10],
    'order_ratio': [0.5]
})

prediction = rf_model.predict(new_order)[0]

print("Example Prediction:")
print(f"  Input: Hour=10, Day=Tuesday, DaysSincePrior=7")
print(f"  Predicted cart size: {prediction:.1f} products")

## Summary

In [ ]:
print("="*60)
print("CART SIZE PREDICTION MODEL - SUMMARY")
print("="*60)
print(f"""
Objective: Predict the number of products in a customer's cart

Features (12 total):
  - order_hour_of_day, order_dow, days_since_prior_order
  - order_number, reordered, order_frequency, avg_basket_size
  - is_weekend, department_diversity, aisle_diversity
  - user_total_orders, order_ratio

Best Model: {models[best_idx]}
  - MAE: {mae_scores[best_idx]:.2f} products
  - RMSE: {rmse_scores[best_idx]:.2f}
  - R²: {r2_scores[best_idx]:.4f}
""")